In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

ACC_DIR = "final_accuracies"
OUT_DIR = "rank_plots"
os.makedirs(OUT_DIR, exist_ok=True)

# translator_reuse_cifar100.csv has a different structure (fit_dataset is always
# cifar100, and skip configs are evaluated across multiple datasets) so it gets its
# own plotting cell below instead of going through the generic rank-plot loop.
TRANSLATOR_REUSE_FILES = {
    os.path.join(ACC_DIR, "translator_reuse_cifar100.csv"),
    os.path.join(ACC_DIR, "transfer_learning", "translator_reuse_cifar100.csv"),
}

# Only files with the model,dataset,accuracy_mean,accuracy_std schema (skip the
# transfer_learning subfolder's per-seed/skip-config files, which have a different shape).
csv_files = sorted(glob.glob(os.path.join(ACC_DIR, "*.csv"))) + [
    os.path.join(ACC_DIR, "transfer_learning", "dinov2base_baseline_and_skips.csv"),
    os.path.join(ACC_DIR, "transfer_learning", "translator_reuse_cifar100.csv"),
]
csv_files = [p for p in csv_files if p not in TRANSLATOR_REUSE_FILES]
csv_files

In [ ]:
import matplotlib.ticker as mticker


def plot_model_dataset(df, model, dataset, source_name, out_dir):
    sub = df[(df["model"] == model) & (df["dataset"] == dataset)].copy()
    if sub.empty:
        return None

    is_baseline = sub["approx_layer"] == "[]"
    if not is_baseline.any():
        return None
    baseline_acc = sub.loc[is_baseline, "accuracy_mean"].iloc[0]
    baseline_std = sub.loc[is_baseline, "accuracy_std"].iloc[0]

    non_baseline = sub.loc[~is_baseline].copy()
    if non_baseline.empty:
        return None

    non_baseline = non_baseline.sort_values("accuracy_mean", ascending=False).reset_index(drop=True)
    non_baseline["rank"] = non_baseline.index + 1

    fig, ax = plt.subplots(figsize=(8, 5))

    mean = non_baseline["accuracy_mean"]
    std = non_baseline["accuracy_std"]
    rank = non_baseline["rank"]

    ax.axhspan(baseline_acc - baseline_std, baseline_acc + baseline_std, color="red", alpha=0.1, label="baseline ±1 std")
    ax.axhline(baseline_acc, color="red", linestyle="--", label="baseline")
    ax.plot(rank, mean, marker="o", color="tab:blue", label="accuracy_mean")
    ax.fill_between(rank, mean - std, mean + std, color="tab:blue", alpha=0.2, label="±1 std")

    best_row = non_baseline.iloc[0]
    ax.annotate(
        f"best skip: {best_row['approx_layer']}",
        xy=(best_row["rank"], best_row["accuracy_mean"]),
        xytext=(10, 10),
        textcoords="offset points",
        fontsize=8,
        color="black",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9),
        arrowprops=dict(arrowstyle="->", color="black", lw=0.8),
    )

    ax.set_xlabel("Rank (1 = highest accuracy)")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model} — {dataset}\n({source_name})")
    ax.legend()

    n_ranks = int(rank.max())
    tick_step = 5 if n_ranks > 20 else 1
    ax.xaxis.set_major_locator(mticker.MultipleLocator(tick_step))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%d"))

    fig.tight_layout()

    safe_model = model.replace("/", "_")
    safe_dataset = str(dataset).replace("/", "_")
    safe_source = os.path.splitext(source_name)[0]
    out_path = os.path.join(out_dir, f"{safe_source}__{safe_model}__{safe_dataset}.png")
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path

In [ ]:
saved = []
for path in csv_files:
    source_name = os.path.basename(path)
    df = pd.read_csv(path)
    if not {"model", "dataset", "approx_layer", "accuracy_mean", "accuracy_std"}.issubset(df.columns):
        print(f"skipping {path}: missing required columns")
        continue
    for model in df["model"].unique():
        for dataset in df.loc[df["model"] == model, "dataset"].unique():
            out_path = plot_model_dataset(df, model, dataset, source_name, OUT_DIR)
            if out_path:
                saved.append(out_path)

print(f"Saved {len(saved)} plots to {OUT_DIR}/")
for p in saved:
    print(p)

In [ ]:
def approx_layer_to_hs(approx_layer):
    if approx_layer == "[]":
        return None
    # e.g. "[(3, 4)]" -> 3
    return int(approx_layer.split("(")[1].split(",")[0])


def plot_translator_reuse_model_dataset(reuse_df, native_df, model, transfer_dataset, source_name, out_dir):
    reuse_model_df = reuse_df[(reuse_df["model"] == model) & (reuse_df["dataset"] == transfer_dataset)].copy()
    native_model_df = native_df[native_df["model"] == model].copy()
    if reuse_model_df.empty or native_model_df.empty:
        return None

    reuse_model_df["H_s"] = reuse_model_df["approx_layer"].apply(approx_layer_to_hs)
    native_model_df["H_s"] = native_model_df["approx_layer"].apply(approx_layer_to_hs)

    transfer_df = reuse_model_df.dropna(subset=["H_s"]).sort_values("H_s")
    # keep the native (original, non-transfer) line aligned to the same skip configs
    # evaluated in the transfer sweep, so both lines share the same x points
    native_curve_df = native_model_df[
        native_model_df["approx_layer"].isin(transfer_df["approx_layer"])
    ].dropna(subset=["H_s"]).sort_values("H_s")

    if transfer_df.empty or native_curve_df.empty:
        return None

    native_baseline_rows = native_model_df[native_model_df["approx_layer"] == "[]"]
    baseline_acc = native_baseline_rows["accuracy_mean"].iloc[0] if not native_baseline_rows.empty else None
    baseline_std = native_baseline_rows["accuracy_std"].iloc[0] if not native_baseline_rows.empty else None

    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(native_curve_df["H_s"], native_curve_df["accuracy_mean"], marker="o", color=colors[0], label="Original Accuracy")
    ax.fill_between(
        native_curve_df["H_s"],
        native_curve_df["accuracy_mean"] - native_curve_df["accuracy_std"],
        native_curve_df["accuracy_mean"] + native_curve_df["accuracy_std"],
        color=colors[0], alpha=0.2,
    )
    if baseline_acc is not None:
        ax.axhspan(baseline_acc - baseline_std, baseline_acc + baseline_std, color=colors[0], alpha=0.1)
        ax.axhline(baseline_acc, color=colors[0], linestyle="--", alpha=0.7, label="Original baseline")

    ax.plot(transfer_df["H_s"], transfer_df["accuracy_mean"], marker="o", color=colors[1], label=f"Transfer Accuracy: {transfer_dataset}")
    ax.fill_between(
        transfer_df["H_s"],
        transfer_df["accuracy_mean"] - transfer_df["accuracy_std"],
        transfer_df["accuracy_mean"] + transfer_df["accuracy_std"],
        color=colors[1], alpha=0.2,
    )

    best_row = transfer_df.sort_values("accuracy_mean", ascending=False).iloc[0]
    ax.annotate(
        f"best skip: {best_row['approx_layer']}",
        xy=(best_row["H_s"], best_row["accuracy_mean"]),
        xytext=(10, 10),
        textcoords="offset points",
        fontsize=8,
        color="black",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9),
        arrowprops=dict(arrowstyle="->", color="black", lw=0.8),
    )

    ax.set_xlabel("H_s (skip start layer)")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model} — {transfer_dataset}: original vs. cifar100-fit translator\n({source_name})")
    ax.legend()

    n_hs = int(transfer_df["H_s"].max())
    tick_step = 5 if n_hs > 20 else 1
    ax.xaxis.set_major_locator(mticker.MultipleLocator(tick_step))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%d"))

    fig.tight_layout()

    safe_model = model.replace("/", "_")
    safe_dataset = str(transfer_dataset).replace("/", "_")
    safe_source = os.path.splitext(source_name)[0]
    out_path = os.path.join(out_dir, f"{safe_source}__{safe_model}__{safe_dataset}.png")
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path

In [ ]:
translator_reuse_saved = []
path = os.path.join(ACC_DIR, "translator_reuse_cifar100.csv")
source_name = os.path.basename(path)
reuse_df = pd.read_csv(path)
native_dataset = "cifar100"  # the dataset every translator in this file was fit on

# maps each transfer dataset to the file holding its own native (non-transfer-learning)
# skip sweep, i.e. translator fit and evaluated on that same dataset
NATIVE_FILE_BY_DATASET = {
    "imagenet-1k": os.path.join(ACC_DIR, "group1_imagenet1k.csv"),
    "chestxray_pneumonia": os.path.join(ACC_DIR, "group1_chestxray_pneumonia.csv"),
    "chestxray14": os.path.join(ACC_DIR, "group1_chestxray14.csv"),
}
native_dfs = {dataset: pd.read_csv(p) for dataset, p in NATIVE_FILE_BY_DATASET.items()}

for model in reuse_df["model"].unique():
    transfer_datasets = sorted(
        d for d in reuse_df.loc[reuse_df["model"] == model, "dataset"].unique()
        if d != native_dataset
    )
    for transfer_dataset in transfer_datasets:
        native_df = native_dfs.get(transfer_dataset)
        if native_df is None:
            print(f"no native file mapped for dataset {transfer_dataset}, skipping")
            continue
        out_path = plot_translator_reuse_model_dataset(reuse_df, native_df, model, transfer_dataset, source_name, OUT_DIR)
        if out_path:
            translator_reuse_saved.append(out_path)

print(f"Saved {len(translator_reuse_saved)} translator-reuse plots to {OUT_DIR}/")
for p in translator_reuse_saved:
    print(p)

In [ ]:
import numpy as np


def is_single_layer_skip(approx_layer):
    if approx_layer == "[]":
        return False
    start, end = approx_layer.strip("[]()").split(",")
    return int(end) - int(start) == 1


# Does the cifar100-fit translator's own instability (std at that skip depth) predict
# how much the transfer accuracy diverges from the native/original accuracy at that
# same skip depth? Build one row per (model, transfer_dataset, H_s).
# Restricted to single-layer skips since that's all translator_reuse_cifar100.csv has,
# and native files also include multi-layer skips that would collide on the same H_s.
records = []
for model in reuse_df["model"].unique():
    cifar_fit_df = reuse_df[(reuse_df["model"] == model) & (reuse_df["dataset"] == native_dataset) & (reuse_df["approx_layer"] != "[]")].copy()
    cifar_fit_df["H_s"] = cifar_fit_df["approx_layer"].apply(approx_layer_to_hs)
    cifar_fit_std_by_hs = cifar_fit_df.set_index("H_s")["accuracy_std"]

    for transfer_dataset, native_df in native_dfs.items():
        transfer_df = reuse_df[(reuse_df["model"] == model) & (reuse_df["dataset"] == transfer_dataset) & (reuse_df["approx_layer"] != "[]")].copy()
        if transfer_df.empty:
            continue
        transfer_df["H_s"] = transfer_df["approx_layer"].apply(approx_layer_to_hs)

        native_model_df = native_df[
            (native_df["model"] == model) & native_df["approx_layer"].apply(is_single_layer_skip)
        ].copy()
        native_model_df["H_s"] = native_model_df["approx_layer"].apply(approx_layer_to_hs)
        native_acc_by_hs = native_model_df.set_index("H_s")["accuracy_mean"]

        for _, row in transfer_df.iterrows():
            H_s = row["H_s"]
            if H_s not in cifar_fit_std_by_hs.index or H_s not in native_acc_by_hs.index:
                continue
            records.append({
                "model": model,
                "transfer_dataset": transfer_dataset,
                "H_s": H_s,
                "cifar100_fit_std": cifar_fit_std_by_hs.loc[H_s],
                "gap": abs(native_acc_by_hs.loc[H_s] - row["accuracy_mean"]),
            })

corr_df = pd.DataFrame(records)
corr_df.head()

In [ ]:
models = corr_df["model"].unique()
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5), sharey=False)

for ax, model in zip(axes, models):
    model_df = corr_df[corr_df["model"] == model]
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for color, transfer_dataset in zip(colors, sorted(model_df["transfer_dataset"].unique())):
        sub = model_df[model_df["transfer_dataset"] == transfer_dataset]
        ax.scatter(sub["cifar100_fit_std"], sub["gap"], color=color, label=transfer_dataset, alpha=0.8)

    if len(model_df) >= 2:
        r = np.corrcoef(model_df["cifar100_fit_std"], model_df["gap"])[0, 1]
        m, b = np.polyfit(model_df["cifar100_fit_std"], model_df["gap"], 1)
        x_line = np.array([model_df["cifar100_fit_std"].min(), model_df["cifar100_fit_std"].max()])
        ax.plot(x_line, m * x_line + b, color="black", linestyle="--", alpha=0.6)
        ax.annotate(
            f"r = {r:.2f}",
            xy=(0.05, 0.95),
            xycoords="axes fraction",
            fontsize=10,
            color="black",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9),
            va="top",
        )

    ax.set_xlabel("cifar100-fit accuracy_std at this H_s")
    ax.set_ylabel("|original accuracy − transfer accuracy|")
    ax.set_title(model)
    ax.legend(fontsize=8)

fig.suptitle("Does cifar100-fit instability predict transfer accuracy gap?")
fig.tight_layout()

out_path = os.path.join(OUT_DIR, "translator_reuse_cifar100__fit_std_vs_transfer_gap.png")
fig.savefig(out_path, dpi=150)
print(f"Saved {out_path}")